<a href="https://colab.research.google.com/github/vl-18/abstractive-text-summarization/blob/main/SFT_T5_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Fine-tune T5-small on CNN/DailyMail for abstractive text summarization.
Includes evaluation (ROUGE) and a Gradio demo.
"""

# ================================
# 1. Install dependencies (Colab only)
# ================================
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import transformers
    import datasets
    import evaluate
    import gradio as gr
    import accelerate
except ImportError:
    print("Installing required libraries...")
    install("transformers")
    install("datasets")
    install("evaluate")
    install("rouge_score")
    install("gradio")
    install("accelerate")
    print("Done.")

In [2]:
# ================================
# 2. Imports
# ================================
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import evaluate
import torch
import numpy as np

In [3]:
# ================================
# 3. Configuration
# ================================
MODEL_NAME = "t5-small"          # or "facebook/bart-base" (slower, needs more memory)
OUTPUT_DIR = "./my_summarization_model"
LOG_DIR = "./logs"

# Dataset sizes (small for fast training, adjust if you have more time)
TRAIN_SAMPLES = 50000
VALID_SAMPLES = 1000
TEST_SAMPLES = 1000

# Training hyperparameters
EPOCHS = 2
BATCH_SIZE = 4                  # per device; reduce if OOM
LEARNING_RATE = 1e-4
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

In [4]:
# ================================
# 4. Load and subset dataset
# ================================
print("Loading CNN/DailyMail dataset...")
dataset = load_dataset("cnn_dailymail", "3.0.0")

train_dataset = dataset["train"].select(range(min(TRAIN_SAMPLES, len(dataset["train"]))))
valid_dataset = dataset["validation"].select(range(min(VALID_SAMPLES, len(dataset["validation"]))))
test_dataset = dataset["test"].select(range(min(TEST_SAMPLES, len(dataset["test"]))))

print(f"Train size: {len(train_dataset)}, Validation size: {len(valid_dataset)}, Test size: {len(test_dataset)}")

Loading CNN/DailyMail dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train size: 50000, Validation size: 1000, Test size: 1000


In [5]:
# ================================
# 5. Load tokenizer and model
# ================================
print(f"Loading tokenizer and model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# ================================
# 6. Tokenization function
# ================================
def tokenize_function(examples):
    # T5 requires "summarize: " prefix
    inputs = ["summarize: " + doc for doc in examples["article"]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )

    # Tokenize summaries (targets)
    # The 'as_target_tokenizer' method is deprecated and no longer needed.
    # Directly tokenize labels. The tokenizer will handle it correctly.
    labels = tokenizer(
        examples["highlights"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(
    tokenize_function, batched=True, remove_columns=["article", "highlights", "id"]
)
tokenized_valid = valid_dataset.map(
    tokenize_function, batched=True, remove_columns=["article", "highlights", "id"]
)
tokenized_test = test_dataset.map(
    tokenize_function, batched=True, remove_columns=["article", "highlights", "id"]
)

Loading tokenizer and model: t5-small


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Tokenizing datasets...


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
# ================================
# 7. Data collator
# ================================
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ================================
# 8. Training arguments
# ================================
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=500,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    logging_dir=LOG_DIR,
    report_to="none",            # disable wandb / tensorboard
    fp16=torch.cuda.is_available(),   # use mixed precision if GPU
)

# ================================
# 9. Trainer
# ================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=data_collator,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [7]:
# ================================
# 10. Train!
# ================================
print("Starting fine-tuning...")
trainer.train()

# Save the final model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# ================================
# 11. Evaluate with ROUGE
# ================================
rouge = evaluate.load("rouge")

def generate_summaries(dataset_subset, model, tokenizer, num_samples=100):
    """Generate summaries for a subset of the dataset."""
    predictions = []
    references = []
    for i, example in enumerate(dataset_subset.select(range(min(num_samples, len(dataset_subset))))):
        input_text = "summarize: " + example["article"]
        input_ids = tokenizer.encode(
            input_text, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True
        )
        if torch.cuda.is_available():
            input_ids = input_ids.to("cuda")
        summary_ids = model.generate(
            input_ids,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True,
        )
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        predictions.append(summary)
        references.append(example["highlights"])
    return predictions, references

print("Evaluating on test set (100 samples)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
preds, refs = generate_summaries(test_dataset, model, tokenizer, num_samples=100)

rouge_results = rouge.compute(predictions=preds, references=refs)
print("\n=== ROUGE Scores ===")
for key, value in rouge_results.items():
    print(f"{key}: {value:.4f}")

Starting fine-tuning...


Epoch,Training Loss,Validation Loss
1,1.077890,0.821676


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,1.077890,0.821676
2,1.051369,0.819669


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./my_summarization_model
Evaluating on test set (100 samples)...

=== ROUGE Scores ===
rouge1: 0.3249
rouge2: 0.1348
rougeL: 0.2497
rougeLsum: 0.2791


In [8]:
# ============================================
# TEST: Fine-tuned T5 vs Original T5 (no fine-tuning)
# ============================================

import torch
from transformers import T5TokenizerFast, T5ForConditionalGeneration
from datasets import load_dataset
import evaluate
import pandas as pd
from tqdm import tqdm

# 1. Load the test dataset (same as before)
print("Loading test dataset...")
dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")
# Use a small subset for quick testing (e.g., 200 articles)
test_subset = dataset.select(range(200))

# 2. Load tokenizer (shared)
tokenizer = T5TokenizerFast.from_pretrained("t5-small")

# 3. Load original model (pre-trained, no fine-tuning)
print("Loading original T5-small model...")
original_model = T5ForConditionalGeneration.from_pretrained("t5-small")

# 4. Load your fine-tuned model
print("Loading fine-tuned model...")
finetuned_model = T5ForConditionalGeneration.from_pretrained("./my_summarization_model")

# Move both models to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
original_model.to(device)
finetuned_model.to(device)
original_model.eval()
finetuned_model.eval()

# 5. Generation function
def generate_summary(model, article, max_length=128):
    input_text = "summarize: " + article
    input_ids = tokenizer.encode(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            input_ids,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# 6. Run evaluation on test subset
print("Generating summaries for test articles...")
results = []
for i, example in enumerate(tqdm(test_subset)):
    article = example["article"]
    reference = example["highlights"]

    orig_summary = generate_summary(original_model, article)
    ft_summary = generate_summary(finetuned_model, article)

    results.append({
        "article_id": i,
        "article": article[:300] + "...",  # preview
        "reference": reference,
        "original_summary": orig_summary,
        "finetuned_summary": ft_summary
    })

# 7. Compute ROUGE scores for both models
rouge = evaluate.load("rouge")

original_predictions = [r["original_summary"] for r in results]
finetuned_predictions = [r["finetuned_summary"] for r in results]
references = [r["reference"] for r in results]

original_rouge = rouge.compute(predictions=original_predictions, references=references)
finetuned_rouge = rouge.compute(predictions=finetuned_predictions, references=references)

# 8. Display comparison table
print("\n" + "="*60)
print("ROUGE SCORE COMPARISON")
print("="*60)
comparison_df = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Original Model": [original_rouge["rouge1"], original_rouge["rouge2"], original_rouge["rougeL"]],
    "Fine-tuned Model": [finetuned_rouge["rouge1"], finetuned_rouge["rouge2"], finetuned_rouge["rougeL"]],
    "Improvement": [
        finetuned_rouge["rouge1"] - original_rouge["rouge1"],
        finetuned_rouge["rouge2"] - original_rouge["rouge2"],
        finetuned_rouge["rougeL"] - original_rouge["rougeL"]
    ]
})
print(comparison_df.to_string(index=False))

# 9. Show 5 qualitative examples side by side
print("\n" + "="*60)
print("QUALITATIVE EXAMPLES (First 5 test articles)")
print("="*60)

for i in range(min(5, len(results))):
    print(f"\n--- Example {i+1} ---")
    print(f"ARTICLE PREVIEW: {results[i]['article']}")
    print(f"\nREFERENCE SUMMARY: {results[i]['reference']}")
    print(f"\nORIGINAL MODEL SUMMARY: {results[i]['original_summary']}")
    print(f"\nFINETUNED MODEL SUMMARY: {results[i]['finetuned_summary']}")
    print("-"*50)

# 10. (Optional) Save results to CSV for later analysis
import csv
with open("comparison_results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["article_id", "reference", "original_summary", "finetuned_summary"])
    writer.writeheader()
    for r in results:
        writer.writerow({
            "article_id": r["article_id"],
            "reference": r["reference"],
            "original_summary": r["original_summary"],
            "finetuned_summary": r["finetuned_summary"]
        })
print("\nResults saved to comparison_results.csv")

Loading test dataset...
Loading original T5-small model...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading fine-tuned model...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Generating summaries for test articles...


100%|██████████| 200/200 [05:44<00:00,  1.72s/it]



ROUGE SCORE COMPARISON
 Metric  Original Model  Fine-tuned Model  Improvement
ROUGE-1        0.304154          0.312389     0.008235
ROUGE-2        0.114487          0.118754     0.004267
ROUGE-L        0.224151          0.231366     0.007215

QUALITATIVE EXAMPLES (First 5 test articles)

--- Example 1 ---
ARTICLE PREVIEW: (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the cou...

REFERENCE SUMMARY: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

ORIGINAL MODEL SUMMARY: the palestinians signed the ICC's founding Rome Statute in January. the ICC also accepted i

In [9]:
# ================================
# 12. (Optional) Gradio Demo
# ================================
try:
    import gradio as gr

    def summarize(article):
        input_text = "summarize: " + article
        input_ids = tokenizer.encode(
            input_text, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True
        )
        if torch.cuda.is_available():
            input_ids = input_ids.to("cuda")
        summary_ids = model.generate(
            input_ids,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True,
        )
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return summary

    demo = gr.Interface(
        fn=summarize,
        inputs=gr.Textbox(lines=10, label="Article"),
        outputs=gr.Textbox(label="Summary"),
        title="AI Text Summarizer (Fine-tuned T5-small)",
        description="Enter a news article to get a concise summary."
    )
    demo.launch(share=True)   # share=True gives a public link
except Exception as e:
    print(f"Gradio demo could not be launched: {e}")
    print("You can still use the model programmatically.")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://31611ffa4fae258ebd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
